# 03 — Reaction rates, flows, and conservation checks


This notebook demonstrates reaction-rate evaluation, abundance-dependent reaction flows, stoichiometric source terms, and A/Z conservation checks.

In [ ]:
from pathlib import Path
import sys

# Make the local editable package importable when the notebooks are run from the repo.
HERE = Path.cwd()
CANDIDATES = [HERE / "src", HERE.parent / "src", HERE.parent.parent / "src"]
for p in CANDIDATES:
    if (p / "nucnetpy").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nucnetpy as nn
print("nucnetpy version:", nn.__version__)
def build_toy_alpha_network():
    """Build a small alpha-chain toy network for tutorials.

    This is not intended to be a physical helium-burning network.  The rates are
    deliberately simple constants so that the examples run quickly and are easy
    to inspect.
    """
    net = nn.Network()
    for name in ["he4", "be8", "c12"]:
        net.add_species(nn.Species.parse(name))

    r1 = nn.Reaction.from_names(
        ["he4", "he4"], ["be8"],
        constant_rate=2.0e-6,
        q_value=0.092,
        label="toy_2alpha_to_be8",
    )
    r2 = nn.Reaction.from_names(
        ["be8", "he4"], ["c12"],
        constant_rate=5.0e-5,
        q_value=7.367,
        label="toy_be8_alpha_to_c12",
    )
    net.reactions.add(r1)
    net.reactions.add(r2)

    zone = nn.Zone(label=("toy", "0", "0"), properties={"t9": "1.0", "rho": "1e4"})
    zone.set_abundance("he4", 0.25)  # X(he4) = A*Y = 1.0
    zone.set_abundance("be8", 0.0)
    zone.set_abundance("c12", 0.0)
    net.add_zone(zone)
    return net
net = build_toy_alpha_network()
zone = net.zone(0)

## Evaluate rates

The toy reactions use constant rates. The same API also supports ReacLib seven-coefficient fits and tabulated rates.

In [ ]:
rows = []
for r in net.reactions.reactions:
    rows.append({'reaction': r.string, 'label': r.label, 'rate_T9_1': r.rate(1.0, rho=1e4), 'statistical_factor': r.statistical_factor()})
pd.DataFrame(rows)

## ReacLib-style rate fit

A `RateFit` stores the seven ReacLib coefficients. Here we choose coefficients that give a simple temperature-dependent demonstration rate.

In [ ]:
fit = nn.RateFit([-20.0, 0.0, 0.0, 3.0, 0.0, 0.0, 0.0], label='demo')
T9 = np.linspace(0.2, 3.0, 100)
rates = [fit.rate(t) for t in T9]
plt.plot(T9, rates)
plt.xlabel('T9')
plt.ylabel('rate')
plt.yscale('log');

## Flows and `ydot`

A reaction flow depends on the rate, density, reactant abundances, and the duplicate-reactant statistical factor.

In [ ]:
flows = net.reactions.flows(zone.abundances, t9=zone.temperature9(), rho=zone.density())
ydot = net.reactions.ydot(zone.abundances, t9=zone.temperature9(), rho=zone.density())
print('flows')
for k, v in flows.items():
    print(f'{k:30s} {v:.6e}')
print()
print('ydot')
for k, v in ydot.items():
    print(f'{k:4s} {v:.6e}')

## Conservation checks

For nuclear reactions, total mass number `A` and charge number `Z` should normally be conserved unless leptons or photons are treated outside the nuclear species list.

In [ ]:
for r in net.reactions.reactions:
    ok, dA, dZ = r.conserves_a_z(net.species)
    print(f'{r.string:30s} conserves={ok}  ΔA={dA}  ΔZ={dZ}')